In [ ]:
# import knihoven
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from uncertainties import ufloat as uf
from scipy.optimize import curve_fit
from pathlib import Path

# path config
downloads_dir = Path.home() / "Downloads"
# ujistime se, ze slozka existuje (kdyby nahodou ne)
downloads_dir.mkdir(parents=True, exist_ok=True)
# output_path = downloads_dir / 'muj_graf.png'

In [ ]:
# definice funkci

# nejistoty typu B pro digitalni pristroje
def unc_B_digital(reading, percent_reading, digits, resolution):    # vse musi byt ve spravnych jednotkach; funguje pouze pro jedny podminky zaroven
    """
    Vypocet nejistoty typu B pro digitalni mereni (ponechame krajni)

    reading : float nebo array
        namerena hodnota
    percent_reading : float
        % of reading
    digits : float
        pocet digitu
    resolution : float
        nejmensi dilek
    """

    max_error = abs(reading) * (percent_reading/100) + digits * resolution  # max_error -- krajni nejistota
    u_B = max_error

    return u_B

# nejistota typu A 
def unc_A(data):
    """
    Výpočet nejistoty typu A
    data : list nebo numpy array
         měření
    return : float
        nejistota typu A
    """
    data = np.array(data)
    n = len(data)

    s = np.std(data, ddof=1)
    u_A = s / np.sqrt(n)
    
    return u_A

In [ ]:
# nacteni google tabulek
sheet_id = '1ZebyvJcuBFii-uzZ6osOv-0ed3yA30mExWTFgEetvO8'
gid = '0'
url = f'https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}'

df_odraz = pd.read_csv(url)

In [ ]:
# odraz
phi0 = np.array(df_odraz['phi0'])*np.pi/180
Ip = np.array(df_odraz['Ip'])
Is = np.array(df_odraz['Is'])
Ip0 = np.array(df_odraz['Ip0'].iloc[0])
Is0 = np.array(df_odraz['Is0'].iloc[0])

Rp = Ip / Ip0
Rs = Is / Is0
RN = (Rp+Rs)/2

phiB = (60.00)*np.pi/180

n_n = np.tan(phiB)

n_porov = (((1+Rs**0.5)*(1+Rp**0.5))/((1-Rs**0.5)*(1-Rp**0.5)))**0.5

phi1 = 99.99
rp = -((np.tan(phi0-phi1))/(np.tan(phi0+phi1)))

phi0_fit = np.linspace(0,90,1000)*np.pi/180
phi0 = phi0_fit
sin_phi1 = np.sin(phi0)/n_n
cos_phi1 = (1 - sin_phi1**2)**0.5
rp_graf = (cos_phi1 - n_n*np.cos(phi0))/(cos_phi1 + n_n*np.cos(phi0))
rs_graf = (np.cos(phi0) - n_n*cos_phi1)/(np.cos(phi0) + n_n*cos_phi1)
rN_graf = (rp_graf**2 + rs_graf**2)/2

fig, ax = plt.subplots(figsize=(16, 9))
ax.set_ylabel(fr"$Odrazivost$", fontsize=20)
ax.set_xlabel(fr"$\varphi_0\,[°]$", fontsize=20)
ax.plot(phi0_fit*180/np.pi, rp_graf**2, label='rp')
ax.plot(phi0_fit*180/np.pi, rs_graf**2, label='rs')
ax.plot(phi0_fit*180/np.pi, rN_graf, label='rs')
ax.tick_params(labelsize=15)
ax.grid(True, alpha=0.7)
ax.legend(fontsize=15)
path_odr = downloads_dir / 'odrazivost.png'
plt.savefig(path_odr, dpi=300, bbox_inches='tight')

In [37]:
# nacteni google tabulek
sheet_id = '1ZebyvJcuBFii-uzZ6osOv-0ed3yA30mExWTFgEetvO8'
gid = '641439416'
url = f'https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}'

df_paral = pd.read_csv(url)

In [ ]:
# planparalelni deska
alpha = np.array(df_paral['alpha'])
x = np.array(df_paral['x'])
d = np.array(df_paral['d'])

d = uf(np.mean(d), unc_A(d))

n_deska = 1*(np.sin(alpha)**2 + ((1-(x/(d*np.sin(alpha))))**(-2))*np.cos(alpha)**2)**0.5
n_deska_fit = 1.6
alpha_fit = np.linspace(0,90,1000)*np.pi/180